# SOLiD-PINN — 01 Paper Reproduction
Run the 18 v0.7.2 simulation stages on Colab CPU. Stage-boundary checkpoints preserve the terminal charge concentration for discharge; an interrupted stage restarts from its beginning. The 11 previously archived stages have metrics but no restart states, so this is a fresh resumable run. Do not interpret the collaborator reference as author-verified experimental data.

## 1. Mount private Google Drive
Place Ref_data_at_C20.csv, Ref_data_at_C10.csv, Ref_data_at_C5.csv and SOLiD-PINN-OCV-Figure4/ocv_charge_digitized.csv and ocv_discharge_digitized.csv under MyDrive/SOLiD-PINN/private_inputs. Use authorized files only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Fetch fixed revision and install dependencies
Use CPU. Keep revision, Python/library versions and input hashes unchanged on resume. If private GitHub access is required, use an approved credential helper, not credentials in cells.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path("/content/SOLiD-PINN")
REV = "aa8a1bb95a444e339d8ecfb45db0661099d2df8f"
ROOT = Path("/content/drive/MyDrive/SOLiD-PINN")
INPUT = ROOT / "private_inputs"
OUTPUT = ROOT / "paper_reproduction_runs"
ENV = {**os.environ, "OPENBLAS_NUM_THREADS":"1", "OMP_NUM_THREADS":"1", "MKL_NUM_THREADS":"1"}
if not (REPO/".git").exists():
    subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","fetch","origin",REV],cwd=REPO,check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
subprocess.run([sys.executable,"-m","pip","install","numpy==2.2.6","scipy==1.15.3","pandas==2.2.3","-e",".[dev]"],cwd=REPO,check=True,env=ENV)

## 3. Verify inputs and tests

In [ ]:
import hashlib
required = [f"Ref_data_at_C{r}.csv" for r in (20,10,5)] + [f"SOLiD-PINN-OCV-Figure4/ocv_{s}_digitized.csv" for s in ("charge","discharge")]
missing = [p for p in required if not (INPUT/p).is_file()]
if missing:
    raise FileNotFoundError(str(missing))
for p in required:
    print(p, hashlib.sha256((INPUT/p).read_bytes()).hexdigest())
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_paper_reproduction.py","tests/test_resumable_reproduction.py"],cwd=REPO,check=True,env=ENV)

## 4. Execute or resume
Start with LIMIT=1, run the execution cell twice (first charge, then resume charge and compute discharge). Verify EXECUTION_STATUS and the RESUMED log. Set LIMIT=None only after the smoke test; failed stages or missing coverage in a full run yield nonzero exit status. Do not run two writers on the same run directory. The runner reuses only checkpoints with matching provenance; changed environments create a new directory.

In [ ]:
CASES = ["literal_0131","inventory_derived","historical_kinetics"]
RATES = [20,10,5]
LIMIT = 1  # Run twice to exercise checkpoint reuse, then set None for full run.
cmd = [sys.executable,"-u","scripts/run_paper_resumable.py",str(INPUT),"--outdir",str(OUTPUT)]
for c in CASES:
    cmd += ["--case",c]
for r in RATES:
    cmd += ["--rate",str(r)]
if LIMIT is not None:
    cmd += ["--limit-new-stages",str(LIMIT)]
RUN_DIR = None
with subprocess.Popen(cmd,cwd=REPO,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
    for line in p.stdout:
        print(line,end="")
        if line.startswith("RUN_DIR "):
            RUN_DIR = Path(line.strip().split(" ",1)[1])
    if p.wait()!=0:
        raise RuntimeError("Runner stopped; inspect logs, keep checkpoints.")
assert RUN_DIR is not None
print(RUN_DIR)

## 5. Inspect completed stages, failed attempts and missing cases
Capacity gate: absolute signed error ≤3.5% and voltage event; report coverage. No new fitting.

In [ ]:
import pandas as pd
from IPython.display import display
if not (RUN_DIR/"metrics.csv").exists():
    raise RuntimeError("No completed stages or recorded failures yet.")
metrics=pd.read_csv(RUN_DIR/"metrics.csv")
display(metrics)
done=metrics[metrics.status.eq("complete")].copy()
expected={(c,r,s) for c in CASES for r in RATES for s in ("charge","discharge")}
observed=set(zip(done.case,done.rate,done.stage))
print(f"Completed: {len(expected & observed)}/{len(expected)}")
print("Remaining:",sorted(expected-observed))
if not done.empty:
    done["capacity_gate"] = done.capacity_error_pct.abs().le(3.5) & done.termination.eq("voltage")
    display(done[["case","rate","stage","termination","capacity_error_pct","rmse_mv","coverage_pct","capacity_gate"]])

In [ ]:
import matplotlib.pyplot as plt
if not done.empty:
    d=done[done.stage.eq("discharge")]
    if not d.empty:
        fig,ax=plt.subplots(figsize=(9,4.5))
        for case,g in d.groupby("case"):
            g=g.sort_values("rate",ascending=False)
            ax.plot(1/g.rate,g.capacity_error_pct,"o-",label=case)
        ax.axhline(3.5,ls="--");ax.axhline(-3.5,ls="--")
        ax.set(xlabel="C-rate (1/h)",ylabel="Signed capacity error (%)",title="Completed discharge stages")
        ax.legend();ax.grid(alpha=.25);fig.tight_layout()
        fig.savefig(RUN_DIR/"capacity_error.png",dpi=180)
        plt.show()

## 6. Handoff
Share reviewed aggregate metrics.csv and manifest.json after checking confidentiality. Keep raw inputs, concentration checkpoints and individual traces private. Update Research Atlas only with verified results. Model C remains a separate future work package.